# Forecasting store sales · Most recent 365 days

For each cutoff, fit the model on the previous 365 calendar days, including the cutoff date. For stores that started selling later, history begins on the first day any family recorded a positive sale there. All subsequent observed zero sales remain in the history.

We forecast daily sales for each store and product family for August 16–31, 2017. The question is whether promotions, holidays, paydays and annual seasonality improve a forecast that already accounts for the day of the week.

For one store and family, the candidate model is

$$
\log(1+\mathrm{sales}_t)=\alpha+\sum_{j=1}^{6}\gamma_j D_{j,t}
+\boldsymbol{\beta}^{\mathsf T}x_t
+\sum_{k=1}^{K}\left[a_k\cos(2\pi kt/365.25)+b_k\sin(2\pi kt/365.25)\right]+u_t .
$$

Monday is the reference weekday. The six $D_{j,t}$ indicators give Tuesday through Sunday their own adjustments to Monday's level. The vector $x_t$ contains the other selected predictors and $\boldsymbol{\beta}$ their coefficients. The Fourier terms model changes through the year, with $t$ measured in calendar days. AutoARIMA chooses an ARIMA model for the remaining variation $u_t$. The coefficients and error orders can differ by store and family. The intercept applies when the error model includes a mean.

The notebook compares 48 predictor lists on three earlier validation periods. The final validation set covers July 31–August 15, 2017.

**Running the notebook.** Place the seven original competition CSVs in `kaggle_comp/data/`, with names such as `train.csv` and `test.csv`. From the repository root, install the project in the Python environment you want to use with `python -m pip install -e ".[dev,kaggle]" joblib`. Select that environment as the notebook kernel. Run the cells in order. Each section saves its results in `kaggle_comp/results/store_sales_365d_bella_settings/`.

In [1]:
from pathlib import Path
from itertools import combinations
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed, parallel_config

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code/backtest/kaggle.py").is_file())
sys.path.insert(0, str(ROOT))
from shared_code.backtest import kaggle as bt

DATA = ROOT / "kaggle_comp/data"
OUT = ROOT / "kaggle_comp/results/store_sales_365d_bella_settings"
OUT.mkdir(parents=True, exist_ok=True)
TRAIN_DAYS = 365  # None uses all available history through the cutoff.
WORKERS = min(4, os.cpu_count() or 1)  # Use 1 to reduce memory use.
SERIES_PER_GROUP = 2  # Four sales groups. Set to None to select predictors on every series.

from importlib.metadata import version
packages = ["numpy", "pandas", "pyarrow", "scipy", "statsforecast", "joblib"]
versions = pd.Series({name: version(name) for name in packages} | {"python": sys.version.split()[0]})
versions.rename("version").to_csv(OUT / "package_versions.csv", index_label="package")

## 1. Build the daily sales table

Each row represents one date, one store, and one product family. We combine the training and test rows before making the calendar variables so that the dates have the same meaning throughout the project. The test rows have empty sales values.

There are four missing Christmas dates in the training file. We add those dates to the calendar and leave their sales missing. This keeps a seven-day lag equal to one calendar week. A reported sale of zero stays zero. A family can sell zero units at an open store, so its sales cannot tell us in advance that the whole store will close.

Both notebooks build the same daily table from the original CSV files and save it to `kaggle_comp/data/store_sales_panel.parquet`. Each training window has its own results folder.

In [2]:
keys = ["store_nbr", "family", "date"]
types = {"id": "Int32", "store_nbr": "int16", "family": "category", "onpromotion": "Int32"}
train = pd.read_csv(DATA / "train.csv", parse_dates=["date"], dtype=types)
test = pd.read_csv(DATA / "test.csv", parse_dates=["date"], dtype=types)
stores = pd.read_csv(DATA / "stores.csv")
events = pd.read_csv(DATA / "holidays_events.csv", parse_dates=["date"])
oil = pd.read_csv(DATA / "oil.csv", parse_dates=["date"])
transactions = pd.read_csv(DATA / "transactions.csv", parse_dates=["date"])

train_end = train["date"].max()
assert test["date"].min() == train_end + pd.Timedelta(days=1)
assert train["sales"].notna().all() and train["sales"].ge(0).all()
sales_rows = pd.concat([train, test], ignore_index=True)
assert sales_rows["id"].is_unique and not sales_rows.duplicated(keys).any()

dates = pd.date_range(train["date"].min(), test["date"].max(), freq="D")
daily_index = pd.MultiIndex.from_product(
    [sorted(sales_rows["store_nbr"].unique()), sorted(sales_rows["family"].unique()), dates],
    names=keys,
)
panel = sales_rows.set_index(keys).reindex(daily_index).reset_index()
panel["family"] = panel["family"].astype("category")
panel["is_test"] = panel["date"].gt(train_end)
assert panel.loc[panel["is_test"], "id"].notna().all()
missing_dates = panel.loc[panel["id"].isna(), "date"].drop_duplicates().dt.date.tolist()
print("Dates added to the calendar:", missing_dates)
del sales_rows, daily_index

Dates added to the calendar: [datetime.date(2013, 12, 25), datetime.date(2014, 12, 25), datetime.date(2015, 12, 25), datetime.date(2016, 12, 25)]


A weekday indicator is 1 on that weekday and 0 otherwise. Monday is the reference day. For the annual pattern, we calculate a sine and a cosine for each harmonic, counting calendar days from the start of the data. Promotions are available for the test dates. For the added calendar dates only, the promotion predictor uses 0 as a placeholder because promotion counts and sales are both missing.

The oil column uses the last published price from at least 16 days earlier. For a 16-day forecast, that price is already known on the day we make the forecast. Transactions are included for exploring historical sales. The transaction file ends when training ends, so the models use other predictors to forecast sales.

In [3]:
calendar = pd.DataFrame({"date": dates})
days = (calendar["date"] - dates[0]).dt.days
calendar["trend_years"] = (days / 365.25).astype("float32")
for weekday in range(1, 7):
    calendar[f"dow_{weekday}"] = calendar["date"].dt.weekday.eq(weekday).astype("int8")
for k in (1, 2):
    angle = 2 * np.pi * k * days / 365.25
    calendar[f"sin{k}"] = np.sin(angle).astype("float32")
    calendar[f"cos{k}"] = np.cos(angle).astype("float32")
calendar["payday"] = (
    calendar["date"].dt.day.eq(15) | calendar["date"].dt.is_month_end
).astype("int8")
daily_oil = oil.set_index("date")["dcoilwtico"].reindex(dates).ffill()
calendar["oil_lag16"] = daily_oil.shift(16).to_numpy(dtype="float32")

panel = panel.merge(calendar, on="date", how="left", validate="many_to_one")
assert panel.loc[panel["id"].notna(), "onpromotion"].notna().all()
panel["log_promo"] = np.log1p(panel["onpromotion"].fillna(0).astype("float64")).astype("float32")
panel = panel.merge(transactions, on=["store_nbr", "date"], how="left", validate="many_to_one")
panel["transactions"] = panel["transactions"].astype("Int32")

For each store, an observed holiday counts if it applies to the whole country, the store's state, or its city. We include transferred holidays on their observed dates, as well as bridge and additional holidays. An original holiday date marked as transferred, a make-up workday, or an event alone does not set the holiday indicator to 1.

In [4]:
observed = events.loc[
    events["type"].isin(["Holiday", "Transfer", "Bridge", "Additional"]) & ~events["transferred"]
]
store_holidays = []
for store in stores.itertuples(index=False):
    applies = (
        observed["locale"].eq("National")
        | (observed["locale"].eq("Regional") & observed["locale_name"].eq(store.state))
        | (observed["locale"].eq("Local") & observed["locale_name"].eq(store.city))
    )
    holiday_dates = observed.loc[applies, ["date"]].drop_duplicates()
    store_holidays.append(holiday_dates.assign(store_nbr=store.store_nbr, holiday=1))

panel = panel.merge(
    pd.concat(store_holidays), on=["store_nbr", "date"], how="left", validate="many_to_one"
)
panel["holiday"] = panel["holiday"].fillna(0).astype("int8")

# Use sales across all families to find the first recorded activity at each store.
# Once a store has started, all later dates stay in its history, including zero-sale days.
store_days = panel.groupby(["store_nbr", "date"], sort=True)["sales"].max().gt(0)
store_started = store_days.groupby(level="store_nbr").cummax().rename("store_started")
panel = panel.merge(store_started, on=["store_nbr", "date"], how="left", validate="many_to_one")
assert panel["id"].notna().sum() == len(train) + len(test)
assert panel.loc[panel["is_test"], "sales"].isna().all()
assert panel.groupby(["store_nbr", "family"], observed=True).size().eq(len(dates)).all()

OUT.mkdir(parents=True, exist_ok=True)
panel_path = DATA / "store_sales_panel.parquet"
temporary_path = DATA / f".store_sales_panel_{os.getpid()}.tmp"
panel.to_parquet(temporary_path, index=False)
temporary_path.replace(panel_path)
print(f"{len(panel):,} rows written to {panel_path}")
display(panel.head())
del train, test

3,036,528 rows written to /Users/obser/Documents/Stony_Brook/Courses/2026 Fall/AMS586/Kaggle project/586_Repository/time-series-forecasting/kaggle_comp/data/store_sales_panel.parquet


,store_nbr,family,date,id,sales,onpromotion,is_test,trend_years,dow_1,dow_2,...,sin1,cos1,sin2,cos2,payday,oil_lag16,log_promo,transactions,holiday,store_started
0,1,AUTOMOTIVE,2013-01-01,0,0.0,0,False,0.000000,1,0,...,0.000000,1.000000,0.000000,1.000000,0,NaN,0.0,<NA>,1,False
1,1,AUTOMOTIVE,2013-01-02,1782,2.0,0,False,0.002738,0,1,...,0.017202,0.999852,0.034398,0.999408,0,NaN,0.0,2111,0,True
2,1,AUTOMOTIVE,2013-01-03,3564,3.0,0,False,0.005476,0,0,...,0.034398,0.999408,0.068755,0.997634,0,NaN,0.0,1833,0,True
3,1,AUTOMOTIVE,2013-01-04,5346,3.0,0,False,0.008214,0,0,...,0.051584,0.998669,0.103031,0.994678,0,NaN,0.0,1863,0,True
4,1,AUTOMOTIVE,2013-01-05,7128,5.0,0,False,0.010951,0,0,...,0.068755,0.997634,0.137185,0.990545,0,NaN,0.0,1509,0,True


## Fit one store and product family

The baseline forecasts each weekday using average log sales on the same weekday in the last four weeks. For example, every Monday in the next 16 days uses the same four past Mondays.

We use Bella’s settings from `04_sarimax.ipynb`: a weekly season of 7 days and BIC to compare ARIMA orders. AutoARIMA uses its default search to choose the orders, including whether to take a daily or weekly difference. The six weekday indicators repeat every seven days, so a weekly difference cancels them. The best subset comparison later in the notebook chooses the predictors.

For a store with less than a year of history, we skip its annual Fourier terms. We also skip predictors that cannot be estimated from a particular series. A constant holiday flag, for example, gives no information about the holiday effect.

In [5]:
from statsforecast.models import AutoARIMA
from statsforecast.arima import arima_string
import warnings

def baseline_one(history, future):
    recent = history.loc[history.date > history.date.max() - pd.Timedelta(days=28)]
    recent = recent.dropna(subset=["sales"])
    if recent.empty:
        return np.zeros(len(future))
    logs = np.log1p(recent.sales)
    weekday_means = logs.groupby(recent.date.dt.weekday).mean()
    prediction = future.date.dt.weekday.map(weekday_means).fillna(logs.mean())
    return np.expm1(prediction.to_numpy(dtype=float))

We keep missing Christmas dates on the daily calendar. Only for fitting the model, we fill their log sales using earlier observations of the same weekday. Reported zero sales stay zero. A series with constant sales gets a constant forecast. A short series or a failed numerical fit uses the weekday baseline. We convert a log-sales forecast $z$ back to sales with $\max(0, e^z-1)$. The results record the method, fitted order, predictors used and any messages from the order search.

In [6]:
def forecast_one(history, future, columns):
    observed = history.sales.dropna()
    horizon = len(future)
    info = {"method": "", "order": "", "bic": np.nan, "days": len(history),
            "history_start": history.date.min(), "history_end": history.date.max(),
            "used_columns": "", "note": "", "acf_lag1": np.nan, "acf_lag7": np.nan}

    if observed.empty or observed.nunique() == 1:
        value = 0.0 if observed.empty else float(observed.iloc[-1])
        info.update(method="constant", note="No observed sales" if observed.empty else "Constant sales")
        return np.full(horizon, value), info

    if len(observed) < 35:
        info.update(method="weekday baseline", note="Fewer than five weeks of observed sales")
        return baseline_one(history, future), info

    y = np.log1p(history.sales.to_numpy(dtype=float))
    for i in np.flatnonzero(np.isnan(y)):
        same_day = y[np.arange(i - 7, max(-1, i - 29), -7)]
        same_day = same_day[np.isfinite(same_day)]
        preceding = y[:i][np.isfinite(y[:i])]
        y[i] = same_day.mean() if len(same_day) else (preceding.mean() if len(preceding) else 0.0)

    # Keep columns that add information beyond the intercept and earlier columns.
    design = np.ones((len(y), 1))
    keep = []
    for col in columns:
        if len(history) < 365 and col in {"sin1", "cos1", "sin2", "cos2"}:
            continue
        values = history[col].to_numpy(dtype=float)
        if not np.isfinite(values).all():
            raise ValueError(f"Training predictor {col} has missing or infinite values.")
        trial = np.column_stack((design, values))
        if np.linalg.matrix_rank(trial) == trial.shape[1]:
            keep.append(col)
            design = trial
    info["used_columns"] = ", ".join(keep)
    x_train = history[keep].to_numpy(dtype=float) if keep else None
    x_future = future[keep].to_numpy(dtype=float) if keep else None
    if keep and not np.isfinite(x_future).all():
        raise ValueError("A future predictor has missing or infinite values.")

    model = AutoARIMA(season_length=7, ic="bic")
    try:
        with warnings.catch_warnings(record=True) as fit_warnings:
            warnings.simplefilter("always")
            model.fit(y, X=x_train)
            log_prediction = model.predict(h=horizon, X=x_future)["mean"]
            prediction = np.expm1(np.maximum(0, np.asarray(log_prediction)))
        if len(prediction) != horizon or not np.isfinite(prediction).all():
            raise ValueError("Invalid forecast")
    except (ValueError, ArithmeticError, RuntimeError, np.linalg.LinAlgError) as err:
        info.update(method="weekday baseline", note=f"{type(err).__name__}: {err}")
        return baseline_one(history, future), info

    errors = pd.Series(model.model_["residuals"])
    info.update(method="AutoARIMA", order=arima_string(model.model_).strip(),
                bic=float(model.model_["bic"]), acf_lag1=errors.autocorr(1),
                acf_lag7=errors.autocorr(7),
                note=" | ".join(dict.fromkeys(str(w.message) for w in fit_warnings)))
    return prediction, info

## Choose the predictors by best subset

Weekday indicators stay together in every model. We try every subset of four optional terms: promotions, observed holidays, paydays and a linear trend. For each subset we try zero, one or two annual sine/cosine pairs. This gives $2^4 \times 3=48$ candidates. Oil is available for exploration in the daily table. Only predictors available on the forecast date belong in this list.

Each candidate uses the same dates and series. Running 48 candidates on 1,782 series at three cutoffs would require 256,608 fits. By default we select one shared predictor list on a smaller sample. Using sales from before the first validation period, we divide active series into four groups by average log sales and draw two series from each group. This covers different sales levels, although eight series can miss patterns specific to some families. The final validation and test forecasts use all series.

In [7]:
weekday_cols = [f"dow_{day}" for day in range(1, 7)]
optional_groups = {"promotion": ["log_promo"], "holiday": ["holiday"],
                   "payday": ["payday"], "trend": ["trend_years"]}
candidates = {}
for size in range(len(optional_groups) + 1):
    for subset in combinations(optional_groups, size):
        for k in (0, 1, 2):
            label = " + ".join(["weekdays", *subset, f"K={k}"])
            cols = weekday_cols + [c for group in subset for c in optional_groups[group]]
            candidates[label] = cols + [c for j in range(1, k + 1) for c in (f"sin{j}", f"cos{j}")]

first_cutoff = min(bt.FOLDS.values())
recent = panel.loc[panel.date.between(first_cutoff - pd.Timedelta(days=89), first_cutoff)]
levels = np.log1p(recent["sales"]).groupby([recent.store_nbr, recent.family], observed=True).mean()
active = levels[levels.gt(0)].sort_index().rename("mean_log_sales").reset_index()
active["sales_group"] = pd.qcut(active.mean_log_sales.rank(method="first"), 4, labels=False)
if SERIES_PER_GROUP is None:
    sample = levels.rename("mean_log_sales").reset_index()
else:
    sample = active.groupby("sales_group", group_keys=False).sample(n=SERIES_PER_GROUP, random_state=586)
sample.to_csv(OUT / "selection_series.csv", index=False)
sample_keys = set(sample[bt.K].itertuples(index=False, name=None))
print(f"{len(candidates)} candidates, {len(sample)} series, {len(bt.FOLDS)} validation periods")
display(sample.head(8))

48 candidates, 8 series, 3 validation periods


,store_nbr,family,mean_log_sales,sales_group
1356,44,BOOKS,0.367628,0
934,30,LINGERIE,1.469494,0
1486,48,CELEBRATION,3.003563,1
615,20,LAWN AND GARDEN,1.908783,1
765,25,DELI,4.926997,2
741,24,HOME CARE,5.595599,2
1473,47,PERSONAL CARE,6.634045,3
344,11,MEATS,6.450332,3


For each training cutoff, we forecast all 16 days using history through that cutoff. Forecast functions receive only the date and the selected predictor values for future rows. We attach the actual sales afterward to calculate RMSLE. We use the three earlier validation periods to choose the predictors. We save July 31–August 15 for a final comparison.

In [ ]:
def split_series(frame, cutoff, columns):
    in_history = frame.date.le(cutoff) & frame.store_started
    if TRAIN_DAYS is not None:
        in_history &= frame.date.gt(cutoff - pd.Timedelta(days=TRAIN_DAYS))
    history = frame.loc[in_history, ["date", "sales", *columns]]
    rows = frame.loc[frame.date.between(cutoff + pd.Timedelta(days=1),
                                         cutoff + pd.Timedelta(days=bt.H))]
    if len(rows) != bt.H:
        raise ValueError("Expected 16 future dates for each series.")
    return history, rows, rows[["date", *columns]]


def score_candidate(key, frame, name, columns):
    results = []
    for period, cutoff in bt.FOLDS.items():
        history, rows, future = split_series(frame, cutoff, columns)
        pred, info = forecast_one(history, future, columns)
        score = bt.rmsle(rows.sales, pred)
        results.append(dict(store_nbr=key[0], family=key[1], candidate=name, period=period,
                            rmsle=score, method=info["method"], order=info["order"]))
    return results


fields = ["date", "id", "sales", "store_started", *dict.fromkeys(
    c for cols in candidates.values() for c in cols)]
series = {key: group[fields].reset_index(drop=True)
          for key, group in panel.groupby(bt.K, observed=True, sort=True)}
selection_rows = []
for name, columns in candidates.items():
    with parallel_config(backend="loky", n_jobs=WORKERS, inner_max_num_threads=1):
        batches = Parallel()(delayed(score_candidate)(key, series[key], name, columns)
                             for key in sorted(sample_keys))
    selection_rows.extend(row for batch in batches for row in batch)
    pd.DataFrame(selection_rows).to_csv(OUT / "selection_details.csv", index=False)
    print(f"Finished {len(selection_rows) // (len(sample_keys) * len(bt.FOLDS))}/{len(candidates)}: {name}")

Every series and period contributes 16 rows. Squaring the individual RMSLE values, taking their mean and then taking the square root gives the pooled RMSLE over all forecast rows. We choose the lowest pooled RMSLE, using fewer predictors to break exact ties. Check the method counts as well: a fit that needed the weekday baseline used that forecast in its score.

In [ ]:
details = pd.DataFrame(selection_rows)
assert len(details) == len(candidates) * len(sample_keys) * len(bt.FOLDS)
ranking = (details.assign(msle=details.rmsle**2).groupby("candidate").msle.mean()
           .pow(0.5).rename("selection_rmsle").to_frame())
ranking["predictors"] = [len(candidates[name]) for name in ranking.index]
ranking = ranking.join(pd.crosstab(details.candidate, details.method)).sort_values(
    ["selection_rmsle", "predictors"])
ranking.to_csv(OUT / "selection_scores.csv")
if not details.method.eq("AutoARIMA").any():
    raise RuntimeError("No AutoARIMA fit succeeded. Inspect selection_details.csv before continuing.")
winner = ranking.index[0]
selected_columns = candidates[winner]
print("Selected model:", winner)
display(ranking.head(8))

## Final validation on all stores and families

We now apply the selected predictor list to each series. AutoARIMA chooses the error orders again using only that series' training data. The comparison model averages the last four observations of the same weekday on the log scale. The model and the baseline both forecast all 16 days from the same cutoff.

In [ ]:
def predict_series(key, frame, cutoff, columns):
    history, rows, future = split_series(frame, cutoff, columns)
    pred, info = forecast_one(history, future, columns)
    result = rows[["id", "date", "sales"]].copy()
    result["store_nbr"], result["family"] = key
    result["pred"] = pred
    result["baseline"] = baseline_one(history, future)
    return result, dict(store_nbr=key[0], family=key[1], **info)


def forecast_all(cutoff, stage, columns=None):
    if columns is None:
        columns = selected_columns
    with parallel_config(backend="loky", n_jobs=WORKERS, inner_max_num_threads=1):
        batches = Parallel(verbose=5)(
            delayed(predict_series)(key, frame, cutoff, columns) for key, frame in series.items())
    forecasts = pd.concat([batch[0] for batch in batches], ignore_index=True)
    fits = pd.DataFrame([batch[1] for batch in batches])
    forecasts.to_parquet(OUT / f"{stage}_predictions.parquet", index=False)
    fits.to_csv(OUT / f"{stage}_fits.csv", index=False)
    print(fits.method.value_counts())
    return forecasts


validation = forecast_all(pd.Timestamp("2017-07-30"), "validation")
scores = pd.DataFrame([
    {"model": label, "rmsle": bt.rmsle(validation.sales, validation[col])}
    for label, col in [(winner, "pred"), ("Four-week weekday baseline", "baseline")]
])
scores.to_csv(OUT / "validation_scores.csv", index=False)
display(scores)

## Read the validation results

Day 1 is the first day after the training cutoff. Day 16 is the last day we forecast from that same cutoff. Each row below compares the model and the weekday baseline across stores and families on that day. Lower RMSLE is better. A positive improvement means the model had a lower RMSLE than the baseline.

In [ ]:
horizon_scores = []
for day, (date, rows) in enumerate(validation.groupby("date", sort=True), start=1):
    model_score = bt.rmsle(rows["sales"], rows["pred"])
    baseline_score = bt.rmsle(rows["sales"], rows["baseline"])
    horizon_scores.append({
        "day": day, "date": date, "series": len(rows),
        "model_rmsle": model_score, "baseline_rmsle": baseline_score,
        "improvement": baseline_score - model_score,
    })
horizon_scores = pd.DataFrame(horizon_scores)
horizon_scores.to_csv(OUT / "validation_by_day.csv", index=False)
display(horizon_scores.round(4))

The plot follows one store and family over the same 16 days. The vertical axis shows log(1 + sales), the scale used in RMSLE. A smaller vertical gap between a forecast and actual sales means a smaller error for that day.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

example_store, example_family = 44, "BEVERAGES"
example = validation.loc[
    validation["store_nbr"].eq(example_store) & validation["family"].eq(example_family)
].sort_values("date")
if example.empty:
    # A small practice run may contain a different set of series.
    example_store, example_family = validation[["store_nbr", "family"]].iloc[0]
    example = validation.loc[
        validation["store_nbr"].eq(example_store) & validation["family"].eq(example_family)
    ].sort_values("date")

fig, ax = plt.subplots(figsize=(9, 3.5))
for column, label, style in [
    ("sales", "Actual sales", "o-"), ("pred", "Model", "s-"),
    ("baseline", "Weekday baseline", "--"),
]:
    ax.plot(example["date"], np.log1p(example[column]), style, label=label, markersize=4)
ax.set(title=f"Validation forecasts for store {example_store}, {example_family}",
       xlabel="Validation date", ylabel="log(1 + sales)")
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "validation_example.png", dpi=150)
plt.show()

## Forecast the Kaggle test dates

Refit the selected model using sales through August 15, 2017. Match the forecasts to the IDs in the sample submission. Sales can be fractional in this competition, so we keep the predicted values as decimals.

In [ ]:
test_forecasts = forecast_all(bt.TEST_CUTOFF, "test")
template = pd.read_csv(DATA / "sample_submission.csv")
assert list(template) == ["id", "sales"] and template.id.is_unique
assert test_forecasts.id.is_unique and set(test_forecasts.id) == set(template.id)
submission = template[["id"]].merge(
    test_forecasts[["id", "pred"]], on="id", how="left", validate="one_to_one"
).rename(columns={"pred": "sales"})
assert submission.id.tolist() == template.id.tolist()
assert np.isfinite(submission.sales).all() and submission.sales.ge(0).all()
submission.to_csv(OUT / "submission.csv", index=False)
print(f"Saved {len(submission):,} predictions to {OUT / 'submission.csv'}")
display(submission.head())

## Read the saved results

`selection_scores.csv` ranks the predictor lists on the three earlier validation periods. `selection_series.csv` identifies the sample used for that decision. `validation_scores.csv` compares the selected model with the weekday baseline on all series. `validation_by_day.csv` shows which forecast dates contributed the most error.

The notebook also writes one row per series to `validation_fits.csv` and `test_fits.csv`. The `order` column gives the fitted ARIMA model. The `used_columns` column lists the predictors the series had enough history and variation to estimate. The `acf_lag1` and `acf_lag7` columns measure how strongly the fitted errors are correlated one day and one week apart. Values close to zero indicate little remaining correlation at those two lags.

The file `submission.csv` contains the test predictions. The larger Parquet tables are saved locally when you run the notebook.

## Optional comparison using the same predictors

Best subset selection can choose different predictors for each training window. To compare the effect of the window with the predictor list fixed, use six weekday indicators, promotions and two annual Fourier pairs in both notebooks. AutoARIMA still estimates each model using the available training data. Set the switch below to `True` to repeat this comparison for July 31–August 15. This adds up to 1,782 fits. If the selected model already uses these predictors, we can reuse its forecasts.

In [ ]:
RUN_FIXED_COMPARISON = False

if RUN_FIXED_COMPARISON:
    fixed_cols = weekday_cols + ["log_promo", "sin1", "cos1", "sin2", "cos2"]
    if selected_columns == fixed_cols:
        fixed_validation = validation
        fixed_validation.to_parquet(OUT / "fixed_validation_predictions.parquet", index=False)
        fits = pd.read_csv(OUT / "validation_fits.csv")
        fits.to_csv(OUT / "fixed_validation_fits.csv", index=False)
    else:
        fixed_validation = forecast_all(pd.Timestamp("2017-07-30"), "fixed_validation", fixed_cols)

    fixed_scores = pd.DataFrame([{
        "model": "Weekdays + promotions + two annual Fourier pairs",
        "rmsle": bt.rmsle(fixed_validation.sales, fixed_validation.pred)
    }])
    fixed_scores.to_csv(OUT / "fixed_validation_scores.csv", index=False)
    display(fixed_scores)

## Compare the two training windows

Run this section after both notebooks have produced their validation results. The first table gives the scores for each training window. The best subset rows use the predictors chosen separately by each notebook. The fixed rows use weekdays, promotions and two annual Fourier pairs in both. The fixed rows appear when both fixed-predictor runs are available.

The selection score uses the eight sampled series on three earlier periods. The final validation score uses all 1,782 series on July 31–August 15. Compare the two training windows within the same score column. Smaller RMSLE is better.

In [ ]:
result_folders = {"Recent 365 days": OUT.parent / "store_sales_365d_bella_settings",
                  "All available dates": OUT.parent / "store_sales_all_history_bella_settings"}
both_ready = all((folder / "validation_by_day.csv").is_file() for folder in result_folders.values())
fixed_ready = all((folder / "fixed_validation_scores.csv").is_file() for folder in result_folders.values())

if both_ready:
    rows = []
    for window, folder in result_folders.items():
        scores = pd.read_csv(folder / "validation_scores.csv", float_precision="round_trip").set_index("model").rmsle
        baseline = scores["Four-week weekday baseline"]
        chosen = scores.drop("Four-week weekday baseline")
        choices = [("Best subset", chosen.index[0], chosen.iloc[0])]
        if fixed_ready:
            fixed = pd.read_csv(folder / "fixed_validation_scores.csv", float_precision="round_trip")
            choices.append(("Fixed predictors", "weekdays + promotion + K=2", fixed.rmsle.iloc[0]))

        ranking = pd.read_csv(folder / "selection_scores.csv").set_index("candidate")
        n = pd.read_csv(folder / "validation_by_day.csv").series.sum()
        for method, predictors, final_score in choices:
            rows.append({"training_window": window, "predictor_choice": method, "predictors": predictors,
                         "selection_sample_rmsle": ranking.loc[predictors, "selection_rmsle"],
                         "final_validation_rmsle": final_score, "weekday_baseline_rmsle": baseline,
                         "final_validation_rows": n})

    comparison = pd.DataFrame(rows)
    comparison.to_csv(OUT.parent / "training_window_comparison_bella_settings.csv", index=False)
    display(comparison[["training_window", "predictor_choice", "selection_sample_rmsle",
                        "final_validation_rmsle", "weekday_baseline_rmsle"]].round(5))
else:
    print("Run the other notebook, then rerun this cell to compare the scores.")

### Which product families benefited from recent dates?

The overall score pools all the forecast rows. The table below scores each family separately using its 54 stores and all 16 validation dates. A positive difference means the recent model had a smaller RMSLE. The CSV includes all 33 families. Four families are shown here.

In [ ]:
if all((folder / "validation_predictions.parquet").is_file() for folder in result_folders.values()):
    family_scores = {}
    for window, folder in result_folders.items():
        forecasts = pd.read_parquet(folder / "validation_predictions.parquet",
                                    columns=["family", "sales", "pred"])
        family_scores[window] = {family: bt.rmsle(group.sales, group.pred)
                                 for family, group in forecasts.groupby("family", observed=True)}
    by_family = pd.DataFrame(family_scores).rename(
        columns={"Recent 365 days": "recent_365_rmsle", "All available dates": "all_history_rmsle"})
    by_family["all_minus_recent"] = by_family.all_history_rmsle - by_family.recent_365_rmsle
    by_family = by_family.rename_axis("family").sort_index()
    by_family.to_csv(OUT.parent / "training_window_by_family_bella_settings.csv")
    examples = ["LIQUOR,WINE,BEER", "PRODUCE", "GROCERY I", "DAIRY"]
    display(by_family.loc[by_family.index.intersection(examples, sort=False)].round(4))
else:
    print("Run the final validation section in both notebooks to compare the families.")

The family scores describe July 31–August 15, 2017. Look at the sign and size of the difference to see which training window worked better for each family during those 16 days.